# P5-5 KOSIS 2종 정제 → `clean_kosis_utilization` · `clean_kosis_production_index` 적재

작성 2026-09-19 (담당분배 명세 `new_data/K_Defense_clean전환_담당분배_명세_20260918.md` §1 공통 · §6 5번 — "기획서 미사용이라 마지막"이었으나 사용자 지시로 실행).
규칙 근거는 `docs/reference/data-cleaning-rules.md` §1(6 부분연도 · 9 NULL≠0 · 10 지수↔금액 합산 금지)·§2-11(컬럼 등급 ★/▲/✕),
DDL 은 `db/alter_2026-09-19_kosis_clean.sql`(→ `db/schema.sql` 반영).

| 원본(raw_) | 정제 결과(clean_) | 기대 행 수 | 비고 |
|---|---|---|---|
| `raw_kosis_utilization` 81 | `clean_kosis_utilization` — raw 1:1 | 81 (제외 0) | `in_scope=1` = 통신전자 9행(§2-11 ★), `is_avg_row=1` = 평균 9행 |
| `raw_kosis_production_index` 1,016 | `clean_kosis_production_index` — raw 1:1 | 1,016 (제외 0) | `stat_month` 127개월(2016-01~2026-07), 잠정 `p)` 16행, `scope_grade` ★ 254 / ▲ 254 / ✕ 508 |

원칙
- `raw_`는 읽기만 한다. 대상 `clean_`이 **비어 있을 때만** 적재한다(재적재는 admin `TRUNCATE` 후 재실행).
  제외한 raw 행은 `clean_excluded_row`에 사유와 함께 남겨 `raw = clean + excluded`를 검산한다(이번 2표는 제외 0 기대).
- raw 는 이미 세로형이다(광폭 → 세로형 변환은 `scripts/load_db.py` `frame_kosis_wide1/2`가 raw 적재 때 했다). clean 은 형 변환(연도·월·숫자)과
  등급 플래그(`in_scope`·`scope_grade`)·잠정치 플래그만 더하고 원문은 `value_text`에 보존한다.
- **raw 결함(확인됨)**: `raw_kosis_production_index.stat_ym='p)'` 16행. 원본 1행 헤더 `'M202606 M202606 2026.06 p)'`(잠정 표기 `p)`)를
  `frame_kosis_wide2`가 `split()[-1]`로 잘라 마지막 토큰만 남긴 것. **raw 는 고치지 않고** clean 에서 `source_col_no`로 월을 복원한다(§2).
- 지수·가동률(%)을 금액과 합산·비율 계산하지 않는다. "율" 오용 없음 — 가동률은 KOSIS 원본 통계 명칭 그대로.
- 단계별 건수는 `meta_load_log`에 기록한다(`dataset_key` = `kosis_utilization` / `kosis_production_index`). 접속은 `scripts/load_db.py`의 `connect()`(→ `.env`, `etl_rw`).

실행: 저장소 루트에서 `py -3.14 -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 --ExecutePreprocessor.kernel_name=defense-dashboard notebooks/clean_p5_kosis.ipynb`

In [1]:
import sys, re
from pathlib import Path
from datetime import datetime, date
from decimal import Decimal, InvalidOperation
import pandas as pd
import numpy as np

if hasattr(sys.stdout, "reconfigure"):  # Jupyter 커널(OutStream)에는 없음
    sys.stdout.reconfigure(encoding="utf-8")
ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user   # .env 읽기 · meta_load_log 기록

NB_TAG = "notebooks/clean_p5_kosis.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
NOW = datetime.now().replace(microsecond=0)
def q(sql, params=None):
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT or isinstance(v, pd._libs.missing.NAType): return None
    if isinstance(v, Decimal): return v
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime().date()
    return v
def insert_df(table, df, cols, chunk=2000):
    rows = [tuple(py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), chunk):
        cur.executemany(sql, rows[i:i+chunk])
    return len(rows)
def to_decimal(s, places):
    # 원문 → Decimal(자릿수 고정). 숫자가 아니면 None(NULL ≠ 0). 쉼표 제거만 하고 다른 보정은 하지 않는다.
    if s is None: return None
    t = str(s).strip().replace(",", "")
    if not re.fullmatch(r"-?\d+(\.\d+)?", t): return None
    try:
        return Decimal(t).quantize(Decimal(1).scaleb(-places))
    except InvalidOperation:
        return None

excluded = []   # (table_name, raw_row_id, reason_code, note) — 이번 2표는 0 기대
print("connected as", USER, "| pandas", pd.__version__)
print("대상 clean 현재 행수:", {t: count(t) for t in ("clean_kosis_utilization", "clean_kosis_production_index")})

connected as etl_rw@% | pandas 3.0.5
대상 clean 현재 행수: {'clean_kosis_utilization': 81, 'clean_kosis_production_index': 1016}


## 1. 방산업체 분야별 평균가동률 → `clean_kosis_utilization`

- raw 81행 = 광폭 9행(분야) × 9열(2016~2024). raw 1:1, PK `raw_row_id`. 업무 키 (`source_file`, `sector_name`, `year`) 고유 확인.
- `year` = raw `year`(CHAR(4)) → SMALLINT. `utilization_pct` = `value_text` 숫자화(DECIMAL(5,1)), 숫자가 아니면 NULL + 건수 출력.
- `is_avg_row` = 분야명 `평균`(KOSIS 가 준 전 분야 평균 — 분야 값으로 다시 평균 내지 않는다).
- `in_scope` = `통신전자` 1, 나머지 0 — `data-cleaning-rules.md` §2-11 컬럼 등급(통신전자 행 ★, 다른 분야 ✕ 스코프 밖). 다른 분야 행도 배경 비교용으로 남긴다(제외 아님).
- 하지 않는 것: 가동률(%)을 금액과 합산·비율 계산. 분야 간 가중 평균.

In [2]:
ru = q("SELECT row_id, sector_name, year, value_text, source_file, source_row_no, source_col_no "
       "FROM raw_kosis_utilization ORDER BY row_id")
print("원본 전체:", len(ru), "| 파일:", ru["source_file"].unique().tolist(),
      "| 분야:", ru["sector_name"].nunique(), "| 연도:", sorted(ru["year"].unique().tolist()))

u = ru.copy()
u["sector_name"] = u["sector_name"].astype(str).str.strip()
assert u["year"].str.fullmatch(r"\d{4}").all(), "year 형식 예외"
u["year"] = u["year"].astype(int)
u["is_avg_row"] = (u["sector_name"] == "평균").astype(int)
u["in_scope"] = (u["sector_name"] == "통신전자").astype(int)
u["utilization_pct"] = u["value_text"].map(lambda s: to_decimal(s, 1))
n_nonnum = int(u["utilization_pct"].isna().sum())
assert not u.duplicated(["source_file", "sector_name", "year"]).any(), "업무 키 중복"
u["raw_row_id"] = u["row_id"]
u["cleaned_at"] = NOW
u["cleaned_by"] = USER

print("중복 처리 후:", len(u), "(raw 1:1, 제외 0) | 업무 키 (source_file, sector_name, year) 고유:", u[["source_file", "sector_name", "year"]].drop_duplicates().shape[0])
print("숫자화 실패(NULL):", n_nonnum, "| 값 범위:", u["utilization_pct"].min(), "~", u["utilization_pct"].max())
print("is_avg_row:", int(u["is_avg_row"].sum()), "| in_scope(통신전자):", int(u["in_scope"].sum()))
print("연도별 행 수:", u.groupby("year").size().to_dict())
display(u[u["in_scope"] == 1][["year", "sector_name", "utilization_pct", "value_text", "source_col_no"]].reset_index(drop=True))

원본 전체: 81 | 파일: ['kosis_409_utilization_by_sector_2016_2024.csv'] | 분야: 9 | 연도: ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
중복 처리 후: 81 (raw 1:1, 제외 0) | 업무 키 (source_file, sector_name, year) 고유: 81
숫자화 실패(NULL): 0 | 값 범위: 36.3 ~ 89.0
is_avg_row: 9 | in_scope(통신전자): 9
연도별 행 수: {2016: 9, 2017: 9, 2018: 9, 2019: 9, 2020: 9, 2021: 9, 2022: 9, 2023: 9, 2024: 9}


,year,sector_name,utilization_pct,value_text,source_col_no
0,2016,통신전자,73.4,73.4,2
1,2017,통신전자,70.8,70.8,3
2,2018,통신전자,75.2,75.2,4
3,2019,통신전자,72.3,72.3,5
4,2020,통신전자,73.6,73.6,6
5,2021,통신전자,73.4,73.4,7
6,2022,통신전자,71.7,71.7,8
7,2023,통신전자,89.0,89.0,9
8,2024,통신전자,85.0,85.0,10


## 2. 광공업생산지수 C26 계열 → `clean_kosis_production_index`

- raw 1,016행 = 광폭 4행(C26·C261·C262·C264) × 254값 열(127개월 × T10/T20). raw 1:1, PK `raw_row_id`.
- `industry_code`·`item_code` = 원문 앞 토큰(`C26 전자 부품, …` → `C26`, `T20 생산지수(계절조정)` → `T20`), 이름은 코드 뒤 원문.
- **`stat_month` 복원 규칙**(raw 결함 대응): 광폭 3열 = 2016.01 T10, 4열 = 2016.01 T20(raw 실측), 이후 2열마다 1개월.
  `month_idx = (source_col_no − 3) // 2`, `stat_month = 2016-01-01 + month_idx 개월`. `stat_ym`이 `YYYY.MM`인 행은 파싱값과 복원값이 **전부 일치해야 한다**(검산 출력).
  `stat_ym = 'p)'`인 16행(`source_col_no` 253~256 × 4산업)만 복원값을 쓰고 `is_month_restored=1`. 253·254 → 2026-06, 255·256 → 2026-07.
- `is_provisional` = 원본 헤더 잠정 표기 `p)`가 붙은 행(= 결함 16행, 2026.06·2026.07). raw 에는 잠정 여부 열이 없어 `stat_ym='p)'`가 유일한 흔적이다.
  `db/schema.sql` raw 주석의 "2026.07 잠정"은 실측으로는 **2026.06·2026.07 두 달**이다.
- `is_partial_year` = 2026(1~7월). `index_value` = `value_text` 숫자화(DECIMAL(8,3)), 숫자가 아니면 NULL.
- `scope_grade`(§2-11 컬럼 등급): `00 전국` × C26·C261 × T20 = ★, 같은 조합 T10 = ▲, 그 외(C262·C264, 시도별) = ✕. ✕도 행은 남긴다(제외 아님).

In [3]:
rp = q("SELECT row_id, region_name, industry_name, stat_ym, item_name, value_text, source_file, source_row_no, source_col_no "
       "FROM raw_kosis_production_index ORDER BY row_id")
print("원본 전체:", len(rp), "| 파일:", rp["source_file"].unique().tolist(),
      "| source_col_no:", int(rp["source_col_no"].min()), "~", int(rp["source_col_no"].max()),
      "| stat_ym 고유:", rp["stat_ym"].nunique(), "| stat_ym='p)':", int((rp["stat_ym"] == "p)").sum()))

p = rp.copy()
for c in ("region_name", "industry_name", "item_name"):
    p[c] = p[c].astype(str).str.strip()
p["industry_code"] = p["industry_name"].str.split(n=1).str[0]
p["industry_name"] = p["industry_name"].str.split(n=1).str[1].str.strip()
p["item_code"] = p["item_name"].str.split(n=1).str[0]
p["item_name"] = p["item_name"].str.split(n=1).str[1].str.strip()
assert set(p["industry_code"]) == {"C26", "C261", "C262", "C264"}, set(p["industry_code"])
assert set(p["item_code"]) == {"T10", "T20"}, set(p["item_code"])

# stat_month: ① stat_ym 파싱 ② source_col_no 복원 → ①이 있으면 ①·② 일치 검산, 없으면(p) 결함) ② 사용
parsed = pd.to_datetime(p["stat_ym"].where(p["stat_ym"].str.fullmatch(r"\d{4}\.\d{2}")), format="%Y.%m", errors="coerce")
col = p["source_col_no"].astype(int)
month_idx = (col - 3) // 2
restored = pd.to_datetime({"year": 2016 + month_idx // 12, "month": 1 + month_idx % 12, "day": 1})
# 열 홀짝 ↔ 항목(3=T10, 4=T20)도 맞는지
assert ((col % 2 == 1) == (p["item_code"] == "T10")).all(), "열 번호 홀짝과 T10/T20 불일치"
ok_mask = parsed.notna()
mismatch = int((parsed[ok_mask] != restored[ok_mask]).sum())
print("stat_ym 정상 행:", int(ok_mask.sum()), "| 파싱값 ≠ source_col_no 복원값:", mismatch, "(0 이어야 함)")
assert mismatch == 0
p["stat_month"] = restored.dt.date
p["is_month_restored"] = (~ok_mask).astype(int)
p["is_provisional"] = p["is_month_restored"]          # p) 표기가 붙은 행 = 잠정치. raw 에 다른 흔적 없음
p["is_partial_year"] = (restored.dt.year == 2026).astype(int)
p["index_value"] = p["value_text"].map(lambda s: to_decimal(s, 3))
n_nonnum = int(p["index_value"].isna().sum())

star = (p["region_name"] == "00 전국") & p["industry_code"].isin(["C26", "C261"])
p["scope_grade"] = np.where(star & (p["item_code"] == "T20"), "★", np.where(star & (p["item_code"] == "T10"), "▲", "✕"))
assert not p.duplicated(["source_file", "region_name", "industry_code", "stat_month", "item_code"]).any(), "업무 키 중복"
p["raw_row_id"] = p["row_id"]
p["cleaned_at"] = NOW
p["cleaned_by"] = USER

print("중복 처리 후:", len(p), "(raw 1:1, 제외 0) | stat_month 고유:", p["stat_month"].nunique(),
      "|", min(p["stat_month"]), "~", max(p["stat_month"]))
print("복원 행(is_month_restored):", int(p["is_month_restored"].sum()), "| 잠정(is_provisional):", int(p["is_provisional"].sum()),
      "| 2026 부분연도 행:", int(p["is_partial_year"].sum()), "| 숫자화 실패(NULL):", n_nonnum)
print("복원 행의 (source_col_no → stat_month):", p[p["is_month_restored"] == 1].groupby("source_col_no")["stat_month"].agg(lambda s: str(s.iloc[0])).to_dict())
print("scope_grade:", p["scope_grade"].value_counts().to_dict())
display(p[p["is_month_restored"] == 1][["industry_code", "item_code", "stat_ym", "source_col_no", "stat_month", "value_text", "is_provisional"]]
        .sort_values(["source_col_no", "industry_code"]).reset_index(drop=True))

원본 전체: 1016 | 파일: ['kosis_101_production_index_c26_201601_202607.csv'] | source_col_no: 3 ~ 256 | stat_ym 고유: 126 | stat_ym='p)': 16
stat_ym 정상 행: 1000 | 파싱값 ≠ source_col_no 복원값: 0 (0 이어야 함)
중복 처리 후: 1016 (raw 1:1, 제외 0) | stat_month 고유: 127 | 2016-01-01 ~ 2026-07-01
복원 행(is_month_restored): 16 | 잠정(is_provisional): 16 | 2026 부분연도 행: 56 | 숫자화 실패(NULL): 0
복원 행의 (source_col_no → stat_month): {253: '2026-06-01', 254: '2026-06-01', 255: '2026-07-01', 256: '2026-07-01'}
scope_grade: {'✕': 508, '▲': 254, '★': 254}


,industry_code,item_code,stat_ym,source_col_no,stat_month,value_text,is_provisional
0,C26,T10,p),253,2026-06-01,159.7,1
1,C261,T10,p),253,2026-06-01,204.6,1
2,C262,T10,p),253,2026-06-01,65.2,1
3,C264,T10,p),253,2026-06-01,120.5,1
4,C26,T20,p),254,2026-06-01,153.2,1
5,C261,T20,p),254,2026-06-01,190.5,1
6,C262,T20,p),254,2026-06-01,66.3,1
7,C264,T20,p),254,2026-06-01,156.7,1
8,C26,T10,p),255,2026-07-01,153.0,1
9,C261,T10,p),255,2026-07-01,185.3,1


## 3. 적재 (대상이 비어 있을 때만) + `meta_load_log`

단계는 `중복 처리 후`(raw 1:1, 제외 0)와 `관련 후보`(§2-11 등급 ★ 행 수)로 기록한다. 제외 행이 없으므로 `clean_excluded_row`에는 쓰지 않는다.

In [4]:
if count("clean_kosis_utilization") > 0:
    print("clean_kosis_utilization 비어 있지 않음 → 건너뜀 (재적재는 admin TRUNCATE 후)")
else:
    cols = ["raw_row_id", "sector_name", "is_avg_row", "in_scope", "year", "utilization_pct", "value_text", "source_file", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_kosis_utilization", u, cols)
    log_stage(cur, "kosis_utilization", "clean_kosis_utilization", "중복 처리 후",
              "raw 1:1(source_file×분야×연도 고유)", n_ins,
              "제외 0. 광폭→세로형은 raw 적재 때 완료. 숫자화 실패 %d행은 NULL(0 아님)" % int(u["utilization_pct"].isna().sum()),
              f"{NB_TAG} §1", USER)
    log_stage(cur, "kosis_utilization", "clean_kosis_utilization", "관련 후보",
              "in_scope=1 (통신전자, §2-11 ★)", int(u["in_scope"].sum()),
              "다른 8개 분야(평균 포함) 72행은 스코프 밖(✕)이지만 배경 비교용으로 행은 보존", f"{NB_TAG} §1", USER)
    conn.commit(); print("clean_kosis_utilization 적재:", count("clean_kosis_utilization"))

if count("clean_kosis_production_index") > 0:
    print("clean_kosis_production_index 비어 있지 않음 → 건너뜀 (재적재는 admin TRUNCATE 후)")
else:
    cols = ["raw_row_id", "region_name", "industry_code", "industry_name", "stat_month", "is_month_restored", "item_code", "item_name",
            "index_value", "is_provisional", "is_partial_year", "scope_grade", "value_text", "source_file", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_kosis_production_index", p, cols)
    log_stage(cur, "kosis_production_index", "clean_kosis_production_index", "중복 처리 후",
              "raw 1:1(source_file×시도×산업×월×항목 고유)", n_ins,
              "제외 0. raw stat_ym='p)' %d행(2026.06·07 잠정치, load_db.py 헤더 절단 결함)은 source_col_no 로 월 복원(is_month_restored=1). 정상 행 파싱값=복원값 불일치 0"
              % int(p["is_month_restored"].sum()),
              f"{NB_TAG} §2", USER)
    log_stage(cur, "kosis_production_index", "clean_kosis_production_index", "관련 후보",
              "scope_grade=★ (전국×C26·C261×T20 계절조정)", int((p["scope_grade"] == "★").sum()),
              "▲ %d(같은 조합 T10 원지수) · ✕ %d(C262·C264)는 스코프 밖이지만 행 보존. 2026은 1~7월 부분연도(is_partial_year)"
              % (int((p["scope_grade"] == "▲").sum()), int((p["scope_grade"] == "✕").sum())),
              f"{NB_TAG} §2", USER)
    conn.commit(); print("clean_kosis_production_index 적재:", count("clean_kosis_production_index"))

assert len(excluded) == 0, excluded   # 이번 2표는 제외 행이 없다

clean_kosis_utilization 비어 있지 않음 → 건너뜀 (재적재는 admin TRUNCATE 후)
clean_kosis_production_index 비어 있지 않음 → 건너뜀 (재적재는 admin TRUNCATE 후)


## 4. 적재 후 검증 — 검산 `raw = clean + excluded`, 월·연도별 건수, 잠정·등급·스코프 분포

In [5]:
display(q('''
SELECT r.tbl, r.raw_n, c.clean_n, COALESCE(x.excl_n, 0) AS excluded_n, r.raw_n - c.clean_n - COALESCE(x.excl_n, 0) AS gap
FROM (SELECT 'raw_kosis_utilization' tbl, COUNT(*) raw_n FROM raw_kosis_utilization
      UNION ALL SELECT 'raw_kosis_production_index', COUNT(*) FROM raw_kosis_production_index) r
JOIN (SELECT 'raw_kosis_utilization' tbl, COUNT(*) clean_n FROM clean_kosis_utilization
      UNION ALL SELECT 'raw_kosis_production_index', COUNT(*) FROM clean_kosis_production_index) c ON c.tbl = r.tbl
LEFT JOIN (SELECT table_name tbl, COUNT(*) excl_n FROM clean_excluded_row GROUP BY table_name) x ON x.tbl = r.tbl
'''))
# 가동률: 연도별 행 수 · 통신전자 값 · 평균 행
display(q("SELECT year, COUNT(*) rows_, SUM(in_scope) in_scope_n, SUM(is_avg_row) avg_n, "
          "MAX(CASE WHEN in_scope=1 THEN utilization_pct END) comm_elec_pct, MAX(CASE WHEN is_avg_row=1 THEN utilization_pct END) avg_pct "
          "FROM clean_kosis_utilization GROUP BY year ORDER BY year"))
display(q("SELECT in_scope, is_avg_row, COUNT(*) n, SUM(utilization_pct IS NULL) null_n FROM clean_kosis_utilization GROUP BY 1,2"))
# 생산지수: 월 범위·잠정·복원·부분연도, 등급 분포, 연도별 건수
display(q("SELECT COUNT(*) n, COUNT(DISTINCT stat_month) months, MIN(stat_month) m0, MAX(stat_month) m1, SUM(is_provisional) prov, "
          "SUM(is_month_restored) restored, SUM(is_partial_year) partial, SUM(index_value IS NULL) null_n FROM clean_kosis_production_index"))
display(q("SELECT scope_grade, industry_code, item_code, COUNT(*) n FROM clean_kosis_production_index GROUP BY 1,2,3 ORDER BY 1,2,3"))
display(q("SELECT YEAR(stat_month) yr, COUNT(*) rows_, COUNT(DISTINCT stat_month) months, SUM(is_provisional) prov, MAX(is_partial_year) partial "
          "FROM clean_kosis_production_index GROUP BY 1 ORDER BY 1"))
display(q("SELECT stat_month, industry_code, item_code, index_value, value_text, is_provisional FROM clean_kosis_production_index "
          "WHERE is_provisional=1 ORDER BY stat_month, industry_code, item_code"))
display(q("SELECT stat_month, industry_code, index_value FROM clean_kosis_production_index WHERE scope_grade='★' AND stat_month >= '2026-01-01' "
          "ORDER BY stat_month, industry_code"))
display(q("SELECT log_id, dataset_key, table_name, stage, stage_detail, row_count FROM meta_load_log WHERE dataset_key LIKE 'kosis%' ORDER BY log_id"))
conn.close()

,tbl,raw_n,clean_n,excluded_n,gap
0,raw_kosis_utilization,81,81,0,0
1,raw_kosis_production_index,1016,1016,0,0


,year,rows_,in_scope_n,avg_n,comm_elec_pct,avg_pct
0,2016,9,1,1,73.4,68.6
1,2017,9,1,1,70.8,69.2
2,2018,9,1,1,75.2,71.2
3,2019,9,1,1,72.3,72.0
4,2020,9,1,1,73.6,72.9
5,2021,9,1,1,73.4,81.4
6,2022,9,1,1,71.7,75.6
7,2023,9,1,1,89.0,77.0
8,2024,9,1,1,85.0,80.3


,in_scope,is_avg_row,n,null_n
0,0,1,9,0
1,0,0,63,0
2,1,0,9,0


,n,months,m0,m1,prov,restored,partial,null_n
0,1016,127,2016-01-01,2026-07-01,16,16,56,0


,scope_grade,industry_code,item_code,n
0,★,C26,T20,127
1,★,C261,T20,127
2,▲,C26,T10,127
3,▲,C261,T10,127
4,✕,C262,T10,127
5,✕,C262,T20,127
6,✕,C264,T10,127
7,✕,C264,T20,127


,yr,rows_,months,prov,partial
0,2016,96,12,0,0
1,2017,96,12,0,0
2,2018,96,12,0,0
3,2019,96,12,0,0
4,2020,96,12,0,0
5,2021,96,12,0,0
6,2022,96,12,0,0
7,2023,96,12,0,0
8,2024,96,12,0,0
9,2025,96,12,0,0


,stat_month,industry_code,item_code,index_value,value_text,is_provisional
0,2026-06-01,C26,T10,159.700,159.7,1
1,2026-06-01,C26,T20,153.200,153.2,1
2,2026-06-01,C261,T10,204.600,204.6,1
3,2026-06-01,C261,T20,190.500,190.5,1
4,2026-06-01,C262,T10,65.200,65.2,1
5,2026-06-01,C262,T20,66.300,66.3,1
6,2026-06-01,C264,T10,120.500,120.5,1
7,2026-06-01,C264,T20,156.700,156.7,1
8,2026-07-01,C26,T10,153.000,153.0,1
9,2026-07-01,C26,T20,159.000,159.0,1


,stat_month,industry_code,index_value
0,2026-01-01,C26,141.100
1,2026-01-01,C261,168.100
2,2026-02-01,C26,170.600
3,2026-02-01,C261,215.500
4,2026-03-01,C26,158.400
5,2026-03-01,C261,196.600
6,2026-04-01,C26,162.400
7,2026-04-01,C261,202.700
8,2026-05-01,C26,150.800
9,2026-05-01,C261,182.400


,log_id,dataset_key,table_name,stage,stage_detail,row_count
0,29,kosis_utilization,raw_kosis_utilization,원본 전체,kosis_409_utilization_by_sector_2016_2024.csv,81
1,30,kosis_production_index,raw_kosis_production_index,원본 전체,kosis_101_production_index_c26_201601_202607.csv,1016
2,126,kosis_utilization,clean_kosis_utilization,중복 처리 후,raw 1:1(source_file×분야×연도 고유),81
3,127,kosis_utilization,clean_kosis_utilization,관련 후보,"in_scope=1 (통신전자, §2-11 ★)",9
4,128,kosis_production_index,clean_kosis_production_index,중복 처리 후,raw 1:1(source_file×시도×산업×월×항목 고유),1016
5,129,kosis_production_index,clean_kosis_production_index,관련 후보,scope_grade=★ (전국×C26·C261×T20 계절조정),254


## 남은 일 (이 노트북이 하지 않은 것)

- **raw 재적재 없음**: `scripts/load_db.py` `frame_kosis_wide2`는 `\d{4}\.\d{2}` 토큰을 찾도록 고쳤지만 raw 는 다시 적재하지 않았다(원본 동결).
  다음 갱신본(확정치)을 받을 때 새 `source_file`로 적재하면 `stat_ym`이 정상으로 들어온다. raw 에는 잠정 표기 `p)`를 담을 열이 없어 파서가 버린다 —
  잠정 여부는 clean `is_provisional`로만 관리한다(갱신본 적재 때는 헤더 `p)` 유무를 노트북이 다시 판정해야 한다).
- `is_provisional`은 지금 파일에서 `stat_ym='p)'` 결함 행과 같다. 정상 파서로 적재된 갱신본에서는 이 등식이 성립하지 않으므로 규칙을 헤더 기준으로 바꿔야 한다.
- 화면 사용은 보류(기획서 v3 미사용). 쓰게 되면 `scope_grade='★'`·`in_scope=1`만 쓰고, 2026은 부분연도·잠정 라벨을 붙인다. 지수·가동률은 금액과 같은 축에 두지 않는다.
- `db/schema.sql` raw 주석 "2026.07 잠정"은 실측(2026.06·07)과 다르다 — 주석만 정정 대상(raw 데이터 변경 아님).